# Security-Check Ablation Study

Measures what each server-side check in the **revised** ThermoCAPTCHA protocol (`thermocaptcha_protocol.py`, the same module used by `Final_code.ipynb` and the RQ2 notebooks) contributes.

**Method.** We run a fixed battery of 11 attacks (100 attempts each) against:
- the full protocol (baseline), and
- 11 ablated servers, each with exactly **one** check switched off via `ThermoCaptchaServer(disabled_checks={...})`.

Every number below is a measured acceptance count. Nothing is hand-assigned. An honest-traffic control is run for every configuration, so a 0 never comes from a broken pipeline.

How to read the matrix:
- If an attack succeeds only when one check is off, that check is **necessary** for that attack.
- If an attack stays at 0 when a check is off, another layer also stops it (defense in depth).

Note: the previous version of this notebook ablated "UID" and "session" factors with fixed, hand-assigned scores. In the revised protocol the user/row ID is bookkeeping, not a security check, so it is not ablated. Session binding is ablated as `session_binding`.

## Setup

In [1]:
import json, os, shutil, time, base64, random, collections
import pandas as pd
import thermocaptcha_protocol as tp

SOURCE_DB, EVAL_DB = "thermal_captcha.db", "ablation_eval.db"   # fresh copy per run
TEST_IMAGE_PATH = "./25.bmp"
SITE_KEY, WEBSITE_NAME = "28024d2408754883a7eff1db9cea2b00", "microsoft.com"
N = 100                                   # attempts per (configuration, attack)
random.seed(2026)

shutil.copy(SOURCE_DB, EVAL_DB)
tp.ensure_demo_secrets(SITE_KEY)
website = tp.Website(SITE_KEY, WEBSITE_NAME, tp.load_website_secret())
with open(TEST_IMAGE_PATH, "rb") as f:
    IMAGE = f.read()

def new_device():
    return tp.ThermoCaptchaClient(tp.rsa.generate_private_key(public_exponent=65537, key_size=2048))

USER_IP, USER_UA = "192.168.1.100", "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
ATTACKER_IP, ATTACKER_UA = "10.0.0.66", "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36"
user, attacker = new_device(), new_device()

CONFIGS = {"Full protocol": set()}
CONFIGS.update({f"no {name}": {name} for name in tp.ABLATION_CHECKS})
pd.DataFrame({"check": list(tp.ABLATION_CHECKS), "what it enforces": list(tp.ABLATION_CHECKS.values())})

NOTE: generated ephemeral demo secrets for: THERMOCAPTCHA_SERVER_SECRET, THERMOCAPTCHA_WEBSITE_SECRET, THERMOCAPTCHA_SITE_SECRETS[28024d2408754883a7eff1db9cea2b00] (set them in the environment for a real deployment).


,check,what it enforces
0,challenge_record,C and sid must match the server-issued challen...
1,challenge_single_use,each challenge can be verified only once
2,challenge_freshness,challenge must be younger than its TTL (server...
3,capture_signature,client signature over H(C || sid || H(I) || pk_d)
4,website_freshness,website request timestamp within the allowed skew
5,website_mac,HMAC_K_site proof on the website request
6,website_nonce,website request nonce is single use
7,session_binding,sessionID must exist in VerifiedSession for th...
8,key_possession,redemption proof signed by the presented publi...
9,context_binding,F_current = F_token = F_session


## Attack battery

| # | Attack | Adversary action |
|---|---|---|
| A1 | Capture replay | re-submit an accepted capture verbatim |
| A2 | Stale challenge | correctly signed capture for a challenge older than the TTL |
| A3 | Image substitution | modify thermal pixel data in transit (image stays valid), signature unchanged |
| A4 | Challenge tampering | alter `nonce`/`issued_at` in transit, signature unchanged |
| A5 | Website-request replay | re-send a redeemed website→server request verbatim |
| A6 | Stale website request | website request with a 300 s old timestamp |
| A7 | Forged website MAC | party without `K_site` submits a redemption request |
| A8 | Token reuse (same device) | redeem an already-redeemed token through a fresh website request |
| A9 | Token relay (CAPTCHA farm) | another device redeems the token from its own IP/UA/key |
| A10 | Token relay, copied public key | relay spoofing the user's IP/UA and presenting the user's public key |
| A11 | Revoked session | redeem a token whose server-side session was revoked |

In [2]:
def honest_capture(server, now=None):
    ch = server.issue_challenge(SITE_KEY, now=now)
    return ch, user.build_capture_payload(ch, IMAGE)

def honest_token(server):
    _, payload = honest_capture(server)
    res = server.verify_capture(payload, USER_IP, USER_UA)
    assert res.ok, res.reason
    return res

def redeem(server, device, token_resp, ip, ua, site=website, mutate=None):
    up = json.loads(device.build_redemption_payload(token_resp))
    if mutate:
        mutate(up)
    return server.redeem_token(site.build_server_request(json.dumps(up), ip, ua))

def a1_capture_replay(s):
    _, p = honest_capture(s)
    assert s.verify_capture(p, USER_IP, USER_UA).ok
    return s.verify_capture(p, USER_IP, USER_UA)

def a2_stale_challenge(s):
    return s.verify_capture(honest_capture(s, now=time.time() - 130)[1], USER_IP, USER_UA)

def a3_image_substitution(s):
    p = json.loads(honest_capture(s)[1])
    img = bytearray(base64.b64decode(p["thermal_image"]))
    for pos in random.sample(range(1024, len(img)), 64):       # pixel data only; header intact
        img[pos] ^= 0x0F
    p["thermal_image"] = base64.b64encode(bytes(img)).decode()
    return s.verify_capture(json.dumps(p), USER_IP, USER_UA)

def a4_challenge_tampering(s):
    p = json.loads(honest_capture(s)[1])
    p["challenge"]["nonce"] = os.urandom(16).hex()
    p["challenge"]["issued_at"] = int(time.time())
    return s.verify_capture(json.dumps(p), USER_IP, USER_UA)

def a5_website_replay(s):
    tok = honest_token(s).data["response_json"]
    req = website.build_server_request(user.build_redemption_payload(tok), USER_IP, USER_UA)
    assert s.redeem_token(req).ok
    return s.redeem_token(req)

def a6_stale_website(s):
    old = tp.Website(SITE_KEY, WEBSITE_NAME, tp.load_website_secret(), clock=lambda: time.time() - 300)
    return redeem(s, user, honest_token(s).data["response_json"], USER_IP, USER_UA, site=old)

def a7_forged_mac(s):
    rogue = tp.Website(SITE_KEY, WEBSITE_NAME, os.urandom(32).hex().encode())
    return redeem(s, user, honest_token(s).data["response_json"], USER_IP, USER_UA, site=rogue)

def a8_token_reuse(s):
    tok = honest_token(s).data["response_json"]
    assert redeem(s, user, tok, USER_IP, USER_UA).ok
    return redeem(s, user, tok, USER_IP, USER_UA)

def a9_token_relay(s):
    return redeem(s, attacker, honest_token(s).data["response_json"], ATTACKER_IP, ATTACKER_UA)

def a10_copied_pk_relay(s):
    def use_victim_pk(up):
        up["public_key"] = user.public_key_pem              # public, copyable; private key is not
    return redeem(s, attacker, honest_token(s).data["response_json"], USER_IP, USER_UA, mutate=use_victim_pk)

def a11_revoked_session(s):
    res = honest_token(s)
    s.conn.execute("DELETE FROM VerifiedSession WHERE SessionID=?", (res.data["sessionID"],))
    s.conn.commit()
    return redeem(s, user, res.data["response_json"], USER_IP, USER_UA)

ATTACKS = {
    "A1 Capture replay": a1_capture_replay,
    "A2 Stale challenge": a2_stale_challenge,
    "A3 Image substitution": a3_image_substitution,
    "A4 Challenge tampering": a4_challenge_tampering,
    "A5 Website-request replay": a5_website_replay,
    "A6 Stale website request": a6_stale_website,
    "A7 Forged website MAC": a7_forged_mac,
    "A8 Token reuse (same device)": a8_token_reuse,
    "A9 Token relay (farm)": a9_token_relay,
    "A10 Relay, copied public key": a10_copied_pk_relay,
    "A11 Revoked session": a11_revoked_session,
}
print(len(CONFIGS), "configurations x", len(ATTACKS), "attacks x", N, "attempts =",
      len(CONFIGS) * len(ATTACKS) * N, "attack attempts")

12 configurations x 11 attacks x 100 attempts = 13200 attack attempts


## Run

In [3]:
accepted = {}             # (config, attack) -> accepted count
reasons = {}              # (config, attack) -> Counter of outcomes
control = {}              # config -> honest end-to-end acceptances
t0 = time.time()
for cfg, disabled in CONFIGS.items():
    server = tp.ThermoCaptchaServer(EVAL_DB, disabled_checks=disabled)
    # Honest end-to-end control: capture + redemption must succeed in every configuration.
    control[cfg] = sum(redeem(server, user, honest_token(server).data["response_json"], USER_IP, USER_UA).ok
                       for _ in range(N))
    for name, attack in ATTACKS.items():
        outcomes = [attack(server) for _ in range(N)]
        accepted[(cfg, name)] = sum(o.ok for o in outcomes)
        reasons[(cfg, name)] = collections.Counter(o.reason for o in outcomes)
    server.close()
    print(f"{cfg:<28} control={control[cfg]}/{N}  successful attacks="
          f"{sum(accepted[(cfg, a)] for a in ATTACKS)}/{len(ATTACKS) * N}")
print(f"done in {time.time() - t0:.1f}s")
assert all(v == N for v in control.values()), 'honest traffic rejected in some configuration'

Full protocol                control=100/100  successful attacks=0/1100


no challenge_record          control=100/100  successful attacks=0/1100


no challenge_single_use      control=100/100  successful attacks=100/1100


no challenge_freshness       control=100/100  successful attacks=100/1100


no capture_signature         control=100/100  successful attacks=100/1100


no website_freshness         control=100/100  successful attacks=100/1100


... (2 outputs left out)

no session_binding           control=100/100  successful attacks=100/1100


no key_possession            control=100/100  successful attacks=100/1100


no context_binding           control=100/100  successful attacks=100/1100


no token_single_use          control=100/100  successful attacks=100/1100
done in 77.8s


## Results: attack success (%) per configuration

Rows are attacks; columns are configurations (one check disabled per column). Each cell is the percentage of the 100 attempts that the server **accepted**.

In [4]:
matrix = pd.DataFrame({cfg: {a: 100.0 * accepted[(cfg, a)] / N for a in ATTACKS} for cfg in CONFIGS})
matrix.loc["Honest control accepted (%)"] = {cfg: 100.0 * control[cfg] / N for cfg in CONFIGS}
pd.set_option("display.width", 250)
print(matrix.T.round(0).astype(int).to_string())

                         A1 Capture replay  A2 Stale challenge  A3 Image substitution  A4 Challenge tampering  A5 Website-request replay  A6 Stale website request  A7 Forged website MAC  A8 Token reuse (same device)  A9 Token relay (farm)  A10 Relay, copied public key  A11 Revoked session  Honest control accepted (%)
Full protocol                            0                   0                      0                       0                          0                         0                      0                             0                      0                             0                    0                          100
no challenge_record                      0                   0                      0                       0                          0                         0                      0                             0                      0                             0                    0                          100
no challenge_single_use                100 

## Per-check impact
For each disabled check: which attacks start to succeed (that check is the only barrier), and which remain blocked by another layer.

In [5]:
rows = []
for name, desc in tp.ABLATION_CHECKS.items():
    cfg = f"no {name}"
    opened = [a for a in ATTACKS if accepted[(cfg, a)] > 0]
    rows.append({
        "Disabled check": name,
        "Enforces": desc,
        "Attacks that succeed": ", ".join(f"{a.split()[0]} ({accepted[(cfg, a)]}/{N})" for a in opened) or "none",
        "Successful attempts": sum(accepted[(cfg, a)] for a in ATTACKS),
    })
impact = pd.DataFrame(rows)
print(impact.to_string(index=False))

print("\nBaseline (full protocol) successful attacks:",
      sum(accepted[("Full protocol", a)] for a in ATTACKS), f"/ {len(ATTACKS) * N}")

      Disabled check                                                 Enforces Attacks that succeed  Successful attempts
    challenge_record  C and sid must match the server-issued challenge record                 none                    0
challenge_single_use                 each challenge can be verified only once         A1 (100/100)                  100
 challenge_freshness    challenge must be younger than its TTL (server clock)         A2 (100/100)                  100
   capture_signature        client signature over H(C || sid || H(I) || pk_d)         A3 (100/100)                  100
   website_freshness        website request timestamp within the allowed skew         A6 (100/100)                  100
         website_mac                 HMAC_K_site proof on the website request         A7 (100/100)                  100
       website_nonce                      website request nonce is single use                 none                    0
     session_binding sessionID must exis

## Rejection reasons
Which check rejected each attack in the full protocol, and what replaced it when the primary check was disabled.

In [6]:
for a in ATTACKS:
    print(f"{a:<30} full protocol: {dict(reasons[('Full protocol', a)])}")
print()
for name in tp.ABLATION_CHECKS:
    cfg = f"no {name}"
    for a in ATTACKS:
        base = reasons[("Full protocol", a)]
        now = reasons[(cfg, a)]
        if now != base and accepted[(cfg, a)] < N:
            print(f"{cfg:<26} {a:<30} -> {dict(now)}")

A1 Capture replay              full protocol: {'challenge already used (replay)': 100}
A2 Stale challenge             full protocol: {'challenge expired': 100}
A3 Image substitution          full protocol: {'invalid signature': 100}
A4 Challenge tampering         full protocol: {'challenge does not match server record': 100}
A5 Website-request replay      full protocol: {'website request replayed': 100}
A6 Stale website request       full protocol: {'website request expired': 100}
A7 Forged website MAC          full protocol: {'invalid website proof': 100}
A8 Token reuse (same device)   full protocol: {'token replayed (nonce already consumed)': 100}
A9 Token relay (farm)          full protocol: {'context mismatch: F_current != F_token or F_token != F_session': 100}
A10 Relay, copied public key   full protocol: {'invalid redemption proof (key possession)': 100}
A11 Revoked session            full protocol: {'unknown sessionID': 100}

no challenge_record        A4 Challenge tampering    